In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [28]:
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.runnables import RunnableParallel, RunnableBranch, RunnableLambda
from pydantic import BaseModel, Field
from typing import Literal

In [3]:
llm = HuggingFaceEndpoint(
    repo_id="meta-llama/Llama-3.1-8B-Instruct",
    task="text-generation"
)

model = ChatHuggingFace(llm=llm)

In [12]:
class Feedback(BaseModel):
    sentiment: Literal['positive', 'negative'] = Field(description='Give the sentiment of the feedback')

In [13]:
parser1 = StrOutputParser()
parser2 = PydanticOutputParser(pydantic_object=Feedback)

In [18]:
prompt1 = PromptTemplate(
    template='classify the sentiment of the following feedback text into positive or negative \n {feedback} \n {format_instruction}',
    input_variables=['feedback'],
    partial_variables={'format_instruction':parser2.get_format_instructions()}
)

In [19]:
classifier_chain = prompt1 | model | parser2

In [43]:
print(classifier_chain.invoke({'feedback':'This is a terrible smartphone'}).sentiment)

negative


In [44]:
prompt2 = PromptTemplate(
    template='Write an appropriate response to this positive feedback \n {feedback}',
    input_variables=['feedback']
)

In [45]:
prompt3 = PromptTemplate(
    template='Write an appropriate response to this negative feedback \n {feedback}',
    input_variables=['feedback']
)

In [46]:
branch_chain = RunnableBranch(
    (lambda x: x.sentiment == 'positive', prompt2 | model | parser1),
    (lambda x: x.sentiment == 'negative', prompt3 | model | parser1),
    RunnableLambda(lambda x: 'could not find sentiment')
)

In [47]:
chain = classifier_chain | branch_chain

In [48]:
print(chain.invoke({'feedback':'This is a terrible phone'}))

Here’s a professional and constructive response to negative feedback:

---
**Subject:** Thank You for Your Feedback

Dear [Customer's Name],

Thank you for taking the time to share your feedback. We truly appreciate your input, as it helps us improve and better serve our customers.

I’m sorry to hear that you had a negative experience. Your concerns are important to us, and we’d like to understand the situation further to address it properly. If you’re willing, could you please provide more details or contact us at [support email/phone number]? We’d be happy to assist you directly.

Your satisfaction is our priority, and we’ll take steps to ensure this doesn’t happen again. Thank you for being a part of our community—we value your trust and look forward to making things right.

Warm regards,
[Your Name]
[Your Position]
[Company Name]
---

**Key points to include:**
1. **Acknowledgment:** Show gratitude for the feedback.
2. **Empathy:** Apologize sincerely and validate their feelings.
3

In [49]:
chain.get_graph().print_ascii()

    +-------------+      
    | PromptInput |      
    +-------------+      
            *            
            *            
            *            
   +----------------+    
   | PromptTemplate |    
   +----------------+    
            *            
            *            
            *            
  +-----------------+    
  | ChatHuggingFace |    
  +-----------------+    
            *            
            *            
            *            
+----------------------+ 
| PydanticOutputParser | 
+----------------------+ 
            *            
            *            
            *            
       +--------+        
       | Branch |        
       +--------+        
            *            
            *            
            *            
    +--------------+     
    | BranchOutput |     
    +--------------+     
